# 04 Market Validation & Decision

本 Notebook 对市场机会识别模型筛选出的代表性出口市场进行二次商业验证。

模型负责回答：

> 哪些市场在贸易数据层面表现出较高机会？

本阶段进一步回答：

> 这些机会是否具有真实市场基础？企业应采取什么策略？

重点验证市场：

- France：成熟核心市场
- Uzbekistan：高增长机会市场
- Romania：市场升级案例
- Serbia：高增长但高波动市场

验证维度包括：

1. 市场需求规模
2. 中国在当地进口市场中的供应地位
3. 主要供应国与竞争格局
4. 需求增长的持续性
5. 市场风险与进入限制
6. 最终市场优先级与策略建议

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import comtradeapicall
import time

OUTPUT_DIR = Path("outputs")

ranking_2024 = pd.read_csv(
    OUTPUT_DIR / "market_opportunity_ranking_2024.csv"
)

change_2023_2024 = pd.read_csv(
    OUTPUT_DIR / "market_opportunity_change_2023_2024.csv"
)

print("2024 ranking:", ranking_2024.shape)
print("2023-2024 change:", change_2023_2024.shape)

2024 ranking: (197, 30)
2023-2024 change: (194, 12)


In [2]:
_last_api_call_time = 0


def safe_preview_final_data(
    max_retries=5,
    min_interval=2.2,
    **kwargs
):
    """
    带限速与重试机制的 UN Comtrade previewFinalData。
    """

    global _last_api_call_time

    for attempt in range(max_retries):

        elapsed = time.monotonic() - _last_api_call_time

        if elapsed < min_interval:
            time.sleep(min_interval - elapsed)

        try:
            # 这里必须调用原始 API
            df = comtradeapicall.previewFinalData(
                **kwargs
            )

            _last_api_call_time = time.monotonic()

        except Exception:
            _last_api_call_time = time.monotonic()

            if attempt == max_retries - 1:
                raise

            time.sleep(min_interval)
            continue

        if df is not None:
            return df

        if attempt < max_retries - 1:
            wait_seconds = min_interval * (attempt + 1)

            print(
                f"API returned None. "
                f"Retrying in {wait_seconds:.1f}s..."
            )

            time.sleep(wait_seconds)

    raise RuntimeError(
        "UN Comtrade API failed after multiple retries."
    )

In [3]:
focus_markets = [
    "France",
    "Uzbekistan",
    "Romania",
    "Serbia"
]

model_validation_base = (
    ranking_2024[
        ranking_2024["destination"].isin(focus_markets)
    ][
        [
            "opportunity_rank",
            "destination",
            "market_segment",
            "opportunity_score",
            "size_score",
            "growth_score",
            "momentum_score",
            "stability_score",
            "activity_score"
        ]
    ]
    .sort_values("opportunity_rank")
    .reset_index(drop=True)
)

model_validation_base

,opportunity_rank,destination,market_segment,opportunity_score,size_score,growth_score,momentum_score,stability_score,activity_score
0,1,France,成熟核心市场,81.314696,94.387755,87.244898,56.701031,89.285714,64.540816
1,3,Uzbekistan,高增长机会市场,78.001788,81.122449,95.918367,95.876289,27.040816,64.540816
2,4,Romania,成熟核心市场,77.755891,79.591837,91.326531,62.886598,80.102041,64.540816
3,10,Serbia,高增长机会市场,73.810751,68.367347,96.428571,92.268041,28.571429,64.540816


In [4]:
# 查看 4 个验证市场在 Comtrade 中的代码
focus_codes = (
    ranking_2024[
        ranking_2024["destination"].isin(focus_markets)
    ][
        ["destination", "destination_code"]
    ]
    .drop_duplicates()
)

focus_codes

,destination,destination_code
0,France,251
2,Uzbekistan,860
3,Romania,642
9,Serbia,688


In [5]:
france_code = int(
    focus_codes.loc[
        focus_codes["destination"] == "France",
        "destination_code"
    ].iloc[0]
)

print("France reporter code:", france_code)

France reporter code: 251


In [6]:
france_import_2024 = safe_preview_final_data(
    typeCode="C",
    freqCode="A",
    clCode="HS",
    period="2024",
    reporterCode=str(france_code),
    cmdCode="850760",
    flowCode="M",          # Import
    partnerCode=None,      # 所有供应国
    partner2Code=None,
    customsCode=None,
    motCode=None,
    maxRecords=500,
    format_output="JSON",
    aggregateBy=None,
    breakdownMode="classic",
    countOnly=None,
    includeDesc=True
)

print("shape:", france_import_2024.shape)

france_import_2024[
    [
        "reporterDesc",
        "partnerCode",
        "partnerDesc",
        "primaryValue",
        "isAggregate"
    ]
].head(20)

shape: (85, 47)


,reporterDesc,partnerCode,partnerDesc,primaryValue,isAggregate
0,France,0,World,3.453509e+09,True
1,France,4,Afghanistan,4.654200e+01,True
2,France,36,Australia,4.432132e+05,True
3,France,40,Austria,1.465625e+06,True
4,France,50,Bangladesh,2.489480e+02,True
5,France,52,Barbados,7.261691e+03,True
6,France,56,Belgium,7.406114e+06,True
7,France,76,Brazil,2.483046e+05,True
8,France,100,Bulgaria,4.289799e+04,True
9,France,104,Myanmar,1.141911e+03,True


In [7]:
# 1. France 从全球进口总额
france_world = france_import_2024[
    france_import_2024["partnerDesc"] == "World"
]

france_total_import = france_world["primaryValue"].sum()


# 2. 各真实供应国
france_suppliers = (
    france_import_2024[
        france_import_2024["isAggregate"] == False
    ][
        ["partnerCode", "partnerISO", "partnerDesc", "primaryValue"]
    ]
    .copy()
)

france_suppliers = (
    france_suppliers
    .sort_values("primaryValue", ascending=False)
    .reset_index(drop=True)
)


# 3. 中国供应额及市场份额
china_import = france_suppliers.loc[
    france_suppliers["partnerISO"] == "CHN",
    "primaryValue"
].sum()

china_share = (
    china_import / france_total_import
    if france_total_import > 0
    else np.nan
)

print(f"France total imports: ${france_total_import:,.0f}")
print(f"Imports from China:   ${china_import:,.0f}")
print(f"China market share:   {china_share:.2%}")

France total imports: $3,453,509,105
Imports from China:   $0
China market share:   0.00%


In [8]:
france_top10_suppliers = france_suppliers.head(10).copy()

france_top10_suppliers["market_share"] = (
    france_top10_suppliers["primaryValue"]
    / france_total_import
)

france_top10_suppliers

,partnerCode,partnerISO,partnerDesc,primaryValue,market_share


In [9]:
france_suppliers = (
    france_import_2024[
        france_import_2024["partnerCode"] != 0
    ][
        [
            "partnerCode",
            "partnerISO",
            "partnerDesc",
            "primaryValue"
        ]
    ]
    .copy()
    .sort_values(
        "primaryValue",
        ascending=False
    )
    .reset_index(drop=True)
)

In [10]:
china_import = france_suppliers.loc[
    france_suppliers["partnerISO"] == "CHN",
    "primaryValue"
].sum()

china_share = (
    china_import / france_total_import
)

print(f"France total imports: ${france_total_import:,.0f}")
print(f"Imports from China:   ${china_import:,.0f}")
print(f"China market share:   {china_share:.2%}")

France total imports: $3,453,509,105
Imports from China:   $1,772,930,191
China market share:   51.34%


In [11]:
france_top10_suppliers = france_suppliers.head(10).copy()

france_top10_suppliers["market_share"] = (
    france_top10_suppliers["primaryValue"]
    / france_total_import
)

france_top10_suppliers[
    [
        "partnerDesc",
        "primaryValue",
        "market_share"
    ]
]

,partnerDesc,primaryValue,market_share
0,China,1.772930e+09,0.513371
1,Poland,8.984712e+08,0.260162
2,Germany,2.743823e+08,0.079450
3,Japan,2.458876e+08,0.071199
4,Rep. of Korea,6.158991e+07,0.017834
5,Italy,2.218376e+07,0.006424
6,USA,2.138628e+07,0.006193
7,Netherlands,2.054187e+07,0.005948
8,"China, Hong Kong SAR",1.987458e+07,0.005755
9,"Areas, nes",1.684804e+07,0.004879


In [12]:
# 前3、前4供应国市场份额
cr3 = france_suppliers.head(3)["primaryValue"].sum() / france_total_import
cr4 = france_suppliers.head(4)["primaryValue"].sum() / france_total_import

print(f"CR3: {cr3:.2%}")
print(f"CR4: {cr4:.2%}")

CR3: 85.30%
CR4: 92.42%


In [13]:
france_suppliers["market_share"] = (
    france_suppliers["primaryValue"]
    / france_total_import
)

france_hhi = (
    france_suppliers["market_share"] ** 2
).sum()

print(f"France HHI: {france_hhi:.4f}")

France HHI: 0.3432


In [14]:
def get_import_structure(reporter_code, year):
    df = safe_preview_final_data(
        typeCode="C",
        freqCode="A",
        clCode="HS",
        period=str(year),
        reporterCode=str(reporter_code),
        cmdCode="850760",
        flowCode="M",
        partnerCode=None,
        partner2Code=None,
        customsCode=None,
        motCode=None,
        maxRecords=500,
        format_output="JSON",
        aggregateBy=None,
        breakdownMode="classic",
        countOnly=None,
        includeDesc=True
    )

    # World 总进口
    total_import = df.loc[
        df["partnerCode"] == 0,
        "primaryValue"
    ].sum()

    # 供应国，排除 World
    suppliers = df[
        df["partnerCode"] != 0
    ].copy()

    china_import = suppliers.loc[
        suppliers["partnerISO"] == "CHN",
        "primaryValue"
    ].sum()

    china_share = (
        china_import / total_import
        if total_import > 0
        else np.nan
    )

    return {
        "year": year,
        "total_import": total_import,
        "china_import": china_import,
        "china_share": china_share
    }

In [15]:
france_2023 = get_import_structure(
    france_code,
    2023
)

france_2024 = get_import_structure(
    france_code,
    2024
)

france_trend = pd.DataFrame(
    [france_2023, france_2024]
)

france_trend

,year,total_import,china_import,china_share
0,2023,3.022704e+09,1.410380e+09,0.466595
1,2024,3.453509e+09,1.772930e+09,0.513371


In [16]:
market_growth = (
    france_2024["total_import"]
    / france_2023["total_import"]
    - 1
)

china_growth = (
    france_2024["china_import"]
    / france_2023["china_import"]
    - 1
)

share_change = (
    france_2024["china_share"]
    - france_2023["china_share"]
)

print(f"France market growth: {market_growth:.2%}")
print(f"China exports growth: {china_growth:.2%}")
print(f"China share change: {share_change:.2%}")

France market growth: 14.25%
China exports growth: 25.71%
China share change: 4.68%


In [17]:
## 市场验证函数

def validate_market(market_name, reporter_code):
    results = []

    for year in [2023, 2024]:

        df = safe_preview_final_data(
            typeCode="C",
            freqCode="A",
            clCode="HS",
            period=str(year),
            reporterCode=str(reporter_code),
            cmdCode="850760",
            flowCode="M",
            partnerCode=None,
            partner2Code=None,
            customsCode=None,
            motCode=None,
            maxRecords=500,
            format_output="JSON",
            aggregateBy=None,
            breakdownMode="classic",
            countOnly=None,
            includeDesc=True
        )

        # World = 该市场总进口
        total_import = df.loc[
            df["partnerCode"] == 0,
            "primaryValue"
        ].sum()

        # 排除 World，得到供应国结构
        suppliers = (
            df[
                df["partnerCode"] != 0
            ][
                [
                    "partnerCode",
                    "partnerISO",
                    "partnerDesc",
                    "primaryValue"
                ]
            ]
            .copy()
            .sort_values(
                "primaryValue",
                ascending=False
            )
            .reset_index(drop=True)
        )

        suppliers["market_share"] = (
            suppliers["primaryValue"]
            / total_import
        )

        # 中国
        china_import = suppliers.loc[
            suppliers["partnerISO"] == "CHN",
            "primaryValue"
        ].sum()

        china_share = (
            china_import / total_import
            if total_import > 0
            else np.nan
        )

        # 集中度
        cr3 = suppliers.head(3)["market_share"].sum()
        cr4 = suppliers.head(4)["market_share"].sum()

        hhi = (
            suppliers["market_share"] ** 2
        ).sum()

        results.append(
            {
                "market": market_name,
                "year": year,
                "total_import": total_import,
                "china_import": china_import,
                "china_share": china_share,
                "cr3": cr3,
                "cr4": cr4,
                "hhi": hhi
            }
        )

    summary = pd.DataFrame(results)

    # 2023 -> 2024
    row_2023 = summary.loc[
        summary["year"] == 2023
    ].iloc[0]

    row_2024 = summary.loc[
        summary["year"] == 2024
    ].iloc[0]

    market_growth = (
        row_2024["total_import"]
        / row_2023["total_import"]
        - 1
    )

    china_growth = (
        row_2024["china_import"]
        / row_2023["china_import"]
        - 1
    )

    share_change = (
        row_2024["china_share"]
        - row_2023["china_share"]
    )

    return {
        "summary": summary,
        "market_growth": market_growth,
        "china_growth": china_growth,
        "share_change": share_change
    }

In [18]:
market_codes = {
    "France": 251,
    "Uzbekistan": 860,
    "Romania": 642,
    "Serbia": 688
}

In [19]:
validation_results = {}

for market in [
    "Uzbekistan",
    "Romania",
    "Serbia"
]:
    validation_results[market] = validate_market(
        market,
        market_codes[market]
    )

In [20]:
market_validation_summary = []

for market, result in validation_results.items():

    summary = result["summary"]

    row_2024 = summary.loc[
        summary["year"] == 2024
    ].iloc[0]

    market_validation_summary.append(
        {
            "market": market,
            "total_import_2024":
                row_2024["total_import"],

            "china_import_2024":
                row_2024["china_import"],

            "china_share_2024":
                row_2024["china_share"],

            "market_growth":
                result["market_growth"],

            "china_growth":
                result["china_growth"],

            "share_change":
                result["share_change"],

            "cr3_2024":
                row_2024["cr3"],

            "hhi_2024":
                row_2024["hhi"]
        }
    )

market_validation_summary = pd.DataFrame(
    market_validation_summary
)

market_validation_summary

,market,total_import_2024,china_import_2024,china_share_2024,market_growth,china_growth,share_change,cr3_2024,hhi_2024
0,Uzbekistan,1.057768e+08,1.047467e+08,0.990261,12.727973,14.144929,0.092649,0.994732,0.980632
1,Romania,4.248164e+08,6.962221e+07,0.163888,0.548405,0.789260,0.022061,0.508186,0.123359
2,Serbia,1.817060e+07,1.266505e+07,0.697008,1.097406,1.428283,0.094974,0.812731,0.496076


In [21]:
france_summary = pd.DataFrame([
    {
        "market": "France",
        "total_import_2024": france_2024["total_import"],
        "china_import_2024": france_2024["china_import"],
        "china_share_2024": france_2024["china_share"],
        "market_growth": market_growth,
        "china_growth": china_growth,
        "share_change": share_change,
        "cr3_2024": cr3,
        "hhi_2024": france_hhi
    }
])

validation_all = pd.concat(
    [
        france_summary,
        market_validation_summary
    ],
    ignore_index=True
)

validation_all = validation_all.merge(
    model_validation_base[
        [
            "destination",
            "opportunity_rank",
            "market_segment",
            "opportunity_score"
        ]
    ],
    left_on="market",
    right_on="destination",
    how="left"
)

validation_all

,market,total_import_2024,china_import_2024,china_share_2024,market_growth,china_growth,share_change,cr3_2024,hhi_2024,destination,opportunity_rank,market_segment,opportunity_score
0,France,3.453509e+09,1.772930e+09,0.513371,0.142523,0.257058,0.046775,0.852983,0.343166,France,1,成熟核心市场,81.314696
1,Uzbekistan,1.057768e+08,1.047467e+08,0.990261,12.727973,14.144929,0.092649,0.994732,0.980632,Uzbekistan,3,高增长机会市场,78.001788
2,Romania,4.248164e+08,6.962221e+07,0.163888,0.548405,0.789260,0.022061,0.508186,0.123359,Romania,4,成熟核心市场,77.755891
3,Serbia,1.817060e+07,1.266505e+07,0.697008,1.097406,1.428283,0.094974,0.812731,0.496076,Serbia,10,高增长机会市场,73.810751


In [22]:
strategy_map = {
    "France": "重点深耕 / 防守优势",
    "Uzbekistan": "需求扩张型 / 谨慎验证",
    "Romania": "重点进攻 / 提升份额",
    "Serbia": "选择性拓展 / 控制风险"
}

validation_all["strategy"] = (
    validation_all["market"].map(strategy_map)
)

In [23]:
final_decision_table = validation_all[
    [
        "market",
        "opportunity_rank",
        "market_segment",
        "opportunity_score",
        "total_import_2024",
        "china_share_2024",
        "market_growth",
        "china_growth",
        "share_change",
        "hhi_2024",
        "strategy"
    ]
].copy()

final_decision_table

,market,opportunity_rank,market_segment,opportunity_score,total_import_2024,china_share_2024,market_growth,china_growth,share_change,hhi_2024,strategy
0,France,1,成熟核心市场,81.314696,3.453509e+09,0.513371,0.142523,0.257058,0.046775,0.343166,重点深耕 / 防守优势
1,Uzbekistan,3,高增长机会市场,78.001788,1.057768e+08,0.990261,12.727973,14.144929,0.092649,0.980632,需求扩张型 / 谨慎验证
2,Romania,4,成熟核心市场,77.755891,4.248164e+08,0.163888,0.548405,0.789260,0.022061,0.123359,重点进攻 / 提升份额
3,Serbia,10,高增长机会市场,73.810751,1.817060e+07,0.697008,1.097406,1.428283,0.094974,0.496076,选择性拓展 / 控制风险


In [24]:
final_decision_table = validation_all[
    [
        "market",
        "opportunity_rank",
        "market_segment",
        "opportunity_score",
        "total_import_2024",
        "china_import_2024",
        "china_share_2024",
        "market_growth",
        "china_growth",
        "share_change",
        "hhi_2024",
        "strategy"
    ]
].copy()

In [25]:
china_export_side = (
    ranking_2024[
        ranking_2024["destination"].isin(focus_markets)
    ][
        ["destination", "export_2024"]
    ]
    .rename(
        columns={
            "destination": "market",
            "export_2024": "china_export_reported"
        }
    )
)

final_decision_table = final_decision_table.merge(
    china_export_side,
    on="market",
    how="left"
)

In [26]:
final_decision_table["mirror_ratio"] = (
    final_decision_table["china_import_2024"]
    / final_decision_table["china_export_reported"]
)

final_decision_table[
    [
        "market",
        "china_export_reported",
        "china_import_2024",
        "mirror_ratio",
        "opportunity_rank",
        "strategy"
    ]
]

,market,china_export_reported,china_import_2024,mirror_ratio,opportunity_rank,strategy
0,France,972602996.0,1.772930e+09,1.822871,1,重点深耕 / 防守优势
1,Uzbekistan,146837513.0,1.047467e+08,0.713351,3,需求扩张型 / 谨慎验证
2,Romania,142004192.0,6.962221e+07,0.490283,4,重点进攻 / 提升份额
3,Serbia,36997999.0,1.266505e+07,0.342317,10,选择性拓展 / 控制风险


### Annual vs Monthly Consistency Check

已对 France、Uzbekistan、Romania、Serbia 进行中国出口侧年度数据与
12 个月月度汇总数据核对。

四个市场均得到：

annual_report / monthly_sum = 1.0

说明中国出口侧年度数据与月度汇总完全一致。
后续发现的 bilateral mirror discrepancy 并非由月度汇总方式造成。

该检查属于一次性数据口径验证，为避免重复调用 UN Comtrade API 触发限流，
最终 Notebook 不在每次 Run All 时重复执行该 API 检查。

In [27]:
final_decision_table["mirror_gap_pct"] = (
    final_decision_table["china_import_2024"]
    / final_decision_table["china_export_reported"]
    - 1
) * 100

final_decision_table["mirror_similarity"] = (
    final_decision_table[
        ["china_import_2024", "china_export_reported"]
    ].min(axis=1)
    /
    final_decision_table[
        ["china_import_2024", "china_export_reported"]
    ].max(axis=1)
)

final_decision_table[
    [
        "market",
        "china_export_reported",
        "china_import_2024",
        "mirror_gap_pct",
        "mirror_similarity"
    ]
]

,market,china_export_reported,china_import_2024,mirror_gap_pct,mirror_similarity
0,France,972602996.0,1.772930e+09,82.287141,0.548585
1,Uzbekistan,146837513.0,1.047467e+08,-28.664909,0.713351
2,Romania,142004192.0,6.962221e+07,-50.971719,0.490283
3,Serbia,36997999.0,1.266505e+07,-65.768276,0.342317


数据口径说明： 中国出口方与目标国进口方存在明显 bilateral mirror discrepancy。经年度数据与月度汇总核对，中国出口侧数据内部一致，因此差异主要来自双边贸易统计口径。本文在市场识别阶段统一采用中国出口方数据，在目标市场验证阶段采用进口国报告数据，镜像差异仅作为数据质量与解释风险提示，不纳入机会评分。

In [28]:
decision_reason_map = {
    "France": (
        "2024年进口市场规模约34.5亿美元，市场同比增长约14.3%；"
        "中国供应份额约51.3%，且中国供应增速高于市场整体增速，"
        "说明中国已具备较强市场地位，适合继续深耕并维护优势。"
    ),

    "Uzbekistan": (
        "2024年进口需求出现爆发式增长，中国供应份额约99%；"
        "机会主要来自当地需求扩张，而非进一步抢占竞争对手份额，"
        "应重点验证高速增长是否具有持续性。"
    ),

    "Romania": (
        "2024年进口市场约4.25亿美元，同比增长约54.8%；"
        "中国份额仅约16.4%，但中国供应增长约78.9%，"
        "同时存在市场扩张和份额提升空间，是较典型的增量型机会。"
    ),

    "Serbia": (
        "2024年进口市场同比增长约109.7%，中国份额约69.7%；"
        "增长信号明显，但绝对市场规模约1817万美元，"
        "更适合选择性拓展而非大规模投入。"
    )
}

data_risk_map = {
    "France": (
        "供应结构较集中，中国已占较高份额，未来增量更多依赖市场继续扩张；"
        "同时需关注波兰等主要竞争供应国。"
    ),

    "Uzbekistan": (
        "中国份额接近99%、HHI接近1，且年度增长超过十倍，"
        "存在需求高增长不可持续及单一供应结构风险。"
    ),

    "Romania": (
        "当前中国份额仍较低，说明竞争空间较大；"
        "但也意味着需要进一步识别主要竞争供应国及中国提升份额的可行性。"
    ),

    "Serbia": (
        "市场绝对规模较小、供应集中度较高，"
        "即使增长较快，也需要警惕低基数导致的增长率放大。"
    )
}

In [29]:
final_decision_table["decision_reason"] = (
    final_decision_table["market"]
    .map(decision_reason_map)
)

final_decision_table["data_risk"] = (
    final_decision_table["market"]
    .map(data_risk_map)
)

In [30]:
final_decision_table["mirror_gap_pct"] = (
    final_decision_table["china_import_2024"]
    / final_decision_table["china_export_reported"]
    - 1
) * 100

final_decision_table["mirror_similarity"] = (
    final_decision_table[
        ["china_import_2024", "china_export_reported"]
    ].min(axis=1)
    /
    final_decision_table[
        ["china_import_2024", "china_export_reported"]
    ].max(axis=1)
)

In [31]:
final_market_decision = final_decision_table[
    [
        "market",
        "opportunity_rank",
        "market_segment",
        "opportunity_score",

        "total_import_2024",
        "china_share_2024",
        "market_growth",
        "china_growth",
        "share_change",
        "hhi_2024",

        "strategy",
        "decision_reason",
        "data_risk",

        "mirror_gap_pct",
        "mirror_similarity"
    ]
].copy()

final_market_decision

,market,opportunity_rank,market_segment,opportunity_score,total_import_2024,china_share_2024,market_growth,china_growth,share_change,hhi_2024,strategy,decision_reason,data_risk,mirror_gap_pct,mirror_similarity
0,France,1,成熟核心市场,81.314696,3.453509e+09,0.513371,0.142523,0.257058,0.046775,0.343166,重点深耕 / 防守优势,2024年进口市场规模约34.5亿美元，市场同比增长约14.3%；中国供应份额约51.3%，...,供应结构较集中，中国已占较高份额，未来增量更多依赖市场继续扩张；同时需关注波兰等主要竞争供应国。,82.287141,0.548585
1,Uzbekistan,3,高增长机会市场,78.001788,1.057768e+08,0.990261,12.727973,14.144929,0.092649,0.980632,需求扩张型 / 谨慎验证,2024年进口需求出现爆发式增长，中国供应份额约99%；机会主要来自当地需求扩张，而非进一步...,中国份额接近99%、HHI接近1，且年度增长超过十倍，存在需求高增长不可持续及单一供应结构风险。,-28.664909,0.713351
2,Romania,4,成熟核心市场,77.755891,4.248164e+08,0.163888,0.548405,0.789260,0.022061,0.123359,重点进攻 / 提升份额,2024年进口市场约4.25亿美元，同比增长约54.8%；中国份额仅约16.4%，但中国供应...,当前中国份额仍较低，说明竞争空间较大；但也意味着需要进一步识别主要竞争供应国及中国提升份额的...,-50.971719,0.490283
3,Serbia,10,高增长机会市场,73.810751,1.817060e+07,0.697008,1.097406,1.428283,0.094974,0.496076,选择性拓展 / 控制风险,2024年进口市场同比增长约109.7%，中国份额约69.7%；增长信号明显，但绝对市场规模...,市场绝对规模较小、供应集中度较高，即使增长较快，也需要警惕低基数导致的增长率放大。,-65.768276,0.342317


France：模型排名第 1，但进一步验证发现中国已经占据超过一半进口市场，所以不是“新市场开发”，而是“核心市场深耕”。

Romania：模型排名第 4，验证后发现市场增长约 55%，中国份额却只有约 16%，因此相比 France，它反而具有更明显的新增量拓展空间。

Uzbekistan：模型增长信号极强，但中国份额已经接近 99%，所以重点不是抢份额，而是判断当地需求爆发是否可持续。

Serbia：增长很快、中国优势也明显，但绝对市场规模较小，因此采用选择性拓展而不是重投入。

In [32]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

final_market_decision.to_csv(
    output_dir / "final_market_validation_decision_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Final market decision table exported.")

Final market decision table exported.


## Final Decision Conclusion

本阶段对市场机会模型筛选出的 France、Uzbekistan、Romania 和 Serbia
进行了目标市场进口需求、China market share、市场增长、
供应集中度及镜像贸易数据一致性验证。

### France
France 属于成熟核心市场。2024 年 HS850760 进口市场规模约 34.5 亿美元，
中国供应份额约 51.3%，且中国供应增长快于市场整体增长。
因此其主要策略不是从零开发，而是继续深耕并维护已有优势。

### Romania
Romania 是本次验证中较典型的增量型机会。
2024 年进口市场约 4.25 亿美元，同比增长约 54.8%，
而中国供应份额仅约 16.4%，同时中国供应增速高于市场整体增速。
因此具备市场扩张与份额提升的双重空间，建议重点拓展。

### Uzbekistan
Uzbekistan 表现出极强需求增长，但中国供应份额已经接近 99%，
供应结构高度集中。
因此其机会主要来自当地需求扩张，而非进一步抢占竞争份额。
建议重点验证高速增长的持续性。

### Serbia
Serbia 市场增长较快，中国供应份额较高，
但当前绝对进口规模仍较小。
因此建议进行选择性拓展，并控制低基数和市场规模风险。

### Data Quality Note
中国出口方与目标国进口方之间存在明显的 bilateral mirror discrepancy。
经核对，中国年度出口数据与月度汇总完全一致，
因此本文将镜像差异视为贸易统计口径差异和数据解释风险。

市场识别阶段统一使用中国出口方数据，
目标市场验证阶段使用进口国报告数据，
镜像差异不纳入 Opportunity Score，也不直接决定市场策略。

### Final Framework

最终形成以下市场决策流程：

贸易数据
→ 特征工程
→ KMeans 市场分层
→ Opportunity Score
→ 历史回测与动态监测
→ 目标市场商业验证
→ 市场策略建议

机器学习模型负责缩小市场调查范围，
而最终商业决策仍需结合市场规模、竞争格局、份额空间及后续政策、
物流和准入风险进一步判断。